# Kerr quasinormal modes

This notebook needs only GeneralizedSasakiNakamura (Julia 1.12 or later).

`qnm(a, s, l, m, n)` computes one quasinormal mode (QNM): its frequency, the GSN amplitudes and the excitation factor. The spin `a` comes first here, unlike the radial functions `GSN_radial(s, l, m, a, omega, ...)` and `Teukolsky_radial(s, l, m, a, omega, ...)`. Units are $G=c=M=1$.

An optional sixth argument selects the branch. `ordinary` (the default) is the mode with $\operatorname{Re}\omega>0$; `qnm(0.68, -2, 2, 2, 0, mirror)` returns the mirror mode with $\operatorname{Re}\omega<0$, whose frequency is by default the reflection $-\overline{\omega}$ of the ordinary $(s, \ell, -m, n)$ root at the same spin; only when that root is not accepted (and did not stop at the negative imaginary axis) is the mirror root polished, starting from the reflection.

In [1]:
using GeneralizedSasakiNakamura

In [2]:
ordinary_mode = qnm(0.68, -2, 2, 2, 0)

QuasiNormalMode(
    parameters = (a = 0.68, s = -2, l = 2, m = 2, n = 0)
    omega = 0.52397510429008398 - 0.081512623631199516im
    lambda = 1.6550030612535795 + 0.36026776076186151im
    incidence_amplitude = -2.9652484956329296e-15 + 1.0066747053005698e-14im
    reflection_amplitude = -0.80073261705109955 + 0.029136276807086189im
    incidence_derivative = -2.143807932934116 + 6.7875557748638862im
    excitation_factor = 0.01983394079017144 + 0.10427056860707475im
    formalism = GSN)

`qnm` returns a `QNMResult` when every check passes; it is displayed as `QuasiNormalMode(` above. Otherwise it returns a `QNMEstimate` (finite values that did not pass every accuracy check; `limiting_gate` says which) or a `QNMFailure`; a zero-damping mode at $|a|=1$ gives a `QNMEndpointResult` without amplitudes. Check `ordinary_mode.status` (`:accepted`, `:estimated` or `:failed`) before using the values.

`omega` is the QNM frequency $\omega_q$ and `lambda` the separation constant of the radial Teukolsky equation. `incidence_amplitude` and `reflection_amplitude` are $A^{{\rm GSN},{\rm inc}}_{\ell m\omega_q}$ and $A^{{\rm GSN},{\rm ref}}_{\ell m\omega_q}$ of the `IN` solution of the GSN equation with unit transmission amplitude, $X^{\rm in}\to e^{-ipr_*}$ at the horizon and $X^{\rm in}\to A^{{\rm GSN},{\rm inc}}_{\ell m\omega}e^{-i\omega r_*}+A^{{\rm GSN},{\rm ref}}_{\ell m\omega}e^{+i\omega r_*}$ at infinity, with $p=\omega-m\Omega_H$. The incidence amplitude vanishes at $\omega_q$, so the printed `incidence_amplitude` is a numerical residual. `incidence_derivative` is $\alpha$ and `excitation_factor` is $B$:

$$\alpha^{{\rm GSN}}_{\ell m\omega_q}=\left.\frac{\partial A^{{\rm GSN},{\rm inc}}_{\ell m\omega}}{\partial\omega}\right|_{\omega=\omega_q},\qquad B^{{\rm GSN}}_{\ell m\omega_q}=\frac{A^{{\rm GSN},{\rm ref}}_{\ell m\omega_q}}{2\omega_q\alpha^{{\rm GSN}}_{\ell m\omega_q}}.$$

With `detailed=true` the result also holds the radial solutions at $\omega_q$: `X` and `R` are the `IN` solutions with unit GSN and unit Teukolsky transmission amplitude, and `Y` is the `IN` solution for $s=-2$, the `UP` solution for $s=+2$ and `missing` for other $s$. `X` takes the tortoise coordinate $r_*$, while `Y` and `R` take the Boyer-Lindquist radius $r$.

In [3]:
detailed = qnm(0.68, -2, 2, 2, 0; detailed=true)
r = 10.0
rstar = rstar_from_r(0.68, r)
(X=detailed.X(rstar), Y=detailed.Y(r), R=detailed.R(r))

(X = -1.0392692738424993 - 1.6891143363807735im, Y = 0.6028384381375009 + 0.14047607467470857im, R = 992.3634889462411 - 381.00316036321533im)

When only the frequency is needed, `qnm_frequency` takes the same `(a, s, l, m, n)`; `qnm_sequence` follows one mode along a list of spins.

In [4]:
root = qnm_frequency(0.68, -2, 2, 2, 0)
roots = qnm_sequence([0.0, 0.3, 0.68], -2, 2, 2, 0)
(root.omega, [r.omega for r in roots])

(0.523975104290084 - 0.08151262363119952im, ComplexF64[0.3736716844180418 - 0.08896231568893566im, 0.41952668176385144 - 0.08772927189431194im, 0.523975104290084 - 0.08151262363119957im])

Overtone labels: for $s=-2$, $\ell=2$, $|m|=2$ co-rotating modes ($am>0$ on the ordinary branch, $am<0$ on the mirror branch) the Schwarzschild frequency $-2i$ (overtone 8) splits at $a\neq0$ into a less damped root $8_0$ and a more damped root $8_1$. The default `convention=:overtone`, which follows the Python [`qnm`](https://github.com/duetosymmetry/qnm) package and Cook & Zalutskiy, labels $8_0$ as $n=8$ and gives $8_1$ no label, so every higher overtone keeps its Schwarzschild number. `convention=:complete_spectrum` numbers every root in order of damping, so $8_1$ is $n=9$.

In [5]:
(n8 = qnm_frequency(0.68, -2, 2, 2, 8).omega,
 n8_1 = qnm_frequency(0.68, -2, 2, 2, 9; convention=:complete_spectrum).omega)

(n8 = 0.410769950743262 - 1.5086151268999994im, n8_1 = 0.4091276402272317 - 1.712771299136728im)

Segments: for $s=-2$, $\ell=2$, $m=0$, $n=10$ the sequence that starts at $a=0$ ends on the negative imaginary axis near $a\approx0.391$, and a second piece leaves the axis at $a\approx0.451$; Cook & Zalutskiy (Phys. Rev. D 94, 104074 (2016)) call the two pieces $10_0$ and $10_1$. The default `segment=0` follows the first piece, so at $a=0.5$ it fails with `stop_reason = :negative_imaginary_axis_endpoint`; `segment=1` selects the second piece.

In [6]:
(segment0 = qnm_frequency(0.5, -2, 2, 0, 10).stop_reason,
 segment1 = qnm_frequency(0.5, -2, 2, 0, 10; segment=1).omega)

(segment0 = :negative_imaginary_axis_endpoint, segment1 = 0.05104369416307723 - 2.5823555488023113im)